# 🔬 Autonomous arXiv Paper Digest & Interactive QA Agent
### *End-to-End Demonstration: Query Routing, arXiv Ingestion, PyMuPDF Parsing, Qdrant Retrieval, Executive Briefing & Grounded QA*

This interactive notebook demonstrates the complete autonomous research paper workflow:
- **LangGraph StateGraph** with persistent SQLite checkpointing and dynamic routing
- **PyMuPDF (fitz)** structural multi-column section parsing
- **Qdrant Local** dense vector indexing powered by FastEmbed (`BAAI/bge-small-en-v1.5`)
- **Executive Briefing Generator** using Google Gemini Flash across 7 structured dimensions including mandatory **Explicit Limitations**
- **Grounded QA RAG Loop** with section-level citations and strict **anti-hallucination refusal** for out-of-scope queries
- **Live Interactive Exploration** via `ipywidgets`


---
## 1. Environment Configuration & Setup
Verify environment variables, configure the structured logger, and initialize the agent dependencies.


In [4]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown, Image, HTML
import ipywidgets as widgets

# Ensure project root is in sys.path
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Load environment variables
load_dotenv()
from src.config import settings
from src.logger import setup_logger
from src.agent.graph import build_agent_graph
from src.agent.state import AgentState

logger = setup_logger("notebook_demo")

print("=" * 60)
print("✅ Environment Configured Successfully")
print(f"• LLM Model:         {settings.gemini_model}")
print(f"• Embedding Model:   {settings.embedding_model}")
print(f"• Qdrant Storage:    {settings.qdrant_path}")
print(f"• State DB:          {settings.sqlite_db_path}")
print(f"• Cache Directory:   {settings.cache_dir}")
print("=" * 60)


ModuleNotFoundError: No module named 'ipywidgets'

---
## 2. LangGraph State Machine Architecture
The agent workflow is modeled as a compiled LangGraph `StateGraph`. The diagram below illustrates:
1. `__start__` routing: Dispatches new research queries to `query_understanding` or multi-turn questions to `qa_answer`.
2. `query_understanding` -> `arxiv_retrieval`: Dispatches ID lookup or topic search.
3. Conditional branching: Direct ID jumps to `fetch_parse`; topic search routes to `selection_ranking`; zero results route to `handle_zero_results`.
4. Ingestion loop: `fetch_parse` -> `chunk_embed` -> `summarize` (or `metadata_fallback` if PDF fails).
5. Grounded QA loop: `qa_answer` retrieves relevant chunks from Qdrant Local with strict anti-hallucination checks.


In [ ]:
# Build and compile the agent StateGraph
graph = build_agent_graph()

# Render the StateGraph visually
try:
    png_data = graph.get_graph().draw_mermaid_png()
    display(Image(png_data))
except Exception as e:
    print(f"Mermaid PNG rendering note: {e}. Rendering Mermaid diagram:")
    display(Markdown(f"```mermaid\n{graph.get_graph().draw_mermaid()}\n```"))


---
## 3. Autonomous Ingestion & Pipeline Execution
We execute the agent on a research paper query (`"1706.03762"` - *Attention Is All You Need* or any arXiv topic).
We stream execution updates in real time to observe intermediate node transitions.


In [ ]:
import uuid

# Test paper: Attention Is All You Need
QUERY = "1706.03762"
SESSION_ID = f"notebook-session-{uuid.uuid4().hex[:8]}"

config = {"configurable": {"thread_id": SESSION_ID}}
initial_state = {
    "query": QUERY,
    "session_id": SESSION_ID,
}

print(f"🚀 Executing Agent Graph for query: '{QUERY}' on thread: '{SESSION_ID}'...\n")

node_labels = {
    "query_understanding": "🔍 [1/6] Query Understanding & Intent Classification",
    "arxiv_retrieval": "📚 [2/6] arXiv API Paper Retrieval",
    "selection_ranking": "📊 [3/6] Semantic Candidate Ranking",
    "fetch_parse": "📄 [4/6] PDF Download & PyMuPDF Structural Parsing",
    "metadata_fallback": "⚠️ Metadata Fallback Handler",
    "chunk_embed": "🧩 [5/6] Section-Aware Chunking & Qdrant Embedding",
    "summarize": "📝 [6/6] Executive Briefing Generation",
    "handle_zero_results": "❌ Zero Results Handler",
    "qa_answer": "💬 Grounded QA Answer Generation",
}

for event in graph.stream(initial_state, config=config, stream_mode="updates"):
    for node_name, updates in event.items():
        label = node_labels.get(node_name, f"⚙️ {node_name}")
        print(f"• Finished Node: {label}")

# Retrieve completed state from SQLite checkpointer
snapshot = graph.get_state(config)
full_state = snapshot.values
selected_paper = full_state.get("selected_paper", {})

print(f"\n🎉 Ingestion Completed! Target Paper: '{selected_paper.get('title', 'N/A')}'")


---
## 4. Candidate Papers & Semantic Ranking Inspection
Inspect candidate papers retrieved from arXiv. When multiple candidates are found via topic search,
the agent computes cosine similarity against query embeddings to rank candidates.


In [ ]:
candidates = full_state.get("candidate_papers", [])
selected = full_state.get("selected_paper", {})

if candidates:
    rows = [
        "| Selected | arXiv ID | Title | Authors | Published | Score |",
        "| :---: | :--- | :--- | :--- | :--- | :---: |"
    ]
    for p in candidates:
        is_sel = "⭐ **YES**" if p.get("arxiv_id") == selected.get("arxiv_id") else "No"
        pid = p.get("arxiv_id", "N/A")
        title = p.get("title", "").replace("\n", " ")[:55]
        authors = ", ".join(p.get("authors", [])[:2])
        pub = p.get("published", "")[:10]
        score = f"{p.get('relevance_score', 1.0):.3f}"
        rows.append(f"| {is_sel} | `{pid}` | {title}... | {authors} | {pub} | {score} |")
    display(Markdown("\n".join(rows)))
else:
    display(Markdown(f"**Direct ID Paper Selection:**\n- **ID**: `{selected.get('arxiv_id')}`\n- **Title**: *{selected.get('title')}*\n- **Authors**: {', '.join(selected.get('authors', []))}"))


---
## 5. PyMuPDF Structural Sections & Qdrant Storage Inspection
The agent extracts multi-column PDF layouts into structural sections (Abstract, Introduction, Architecture, etc.).
Each section is chunked with contextual headers and stored in Qdrant Local with FastEmbed BGE-small embeddings.


In [ ]:
from src.vectorstore.qdrant_store import QdrantStore

sections = full_state.get("sections", [])
print(f"📄 Extracted {len(sections)} structural sections via PyMuPDF:\n")

sec_rows = ["| Section Title | Characters | Preview |", "| :--- | :---: | :--- |"]
for s in sections[:6]:
    stitle = s.get("title", "Untitled")
    clen = len(s.get("content", ""))
    preview = s.get("content", "")[:75].replace("\n", " ")
    sec_rows.append(f"| **{stitle}** | {clen:,} | {preview}... |")
display(Markdown("\n".join(sec_rows)))

# Inspect Qdrant Vector Collection
arxiv_id = selected.get("arxiv_id", QUERY)
qstore = QdrantStore()
chunk_count = qstore.count_chunks(arxiv_id=arxiv_id)
print(f"\n📦 Qdrant Collection status: {chunk_count} chunks indexed for `{arxiv_id}`.")

# Sample semantic search directly against Qdrant
sample_hits = qstore.search_chunks(query="multi-head attention mechanism", arxiv_id=arxiv_id, limit=2)
print("\n🔍 Sample Retrieval Hits for 'multi-head attention mechanism':")
for i, hit in enumerate(sample_hits, 1):
    payload = hit.get("payload", {})
    print(f"\n[Hit {i}] Score: {hit.get('score', 0):.4f} | Section: '{payload.get('section', 'N/A')}' (Page {payload.get('page_number', 'N/A')})")
    print(f"Content: {payload.get('content', '')[:200]}...")


---
## 6. Formatted Executive Briefing Display
The briefing covers all 7 evaluation dimensions with special emphasis on **Explicit Limitations** (mandatory per rubric).


In [ ]:
briefing = full_state.get("briefing", {})
markdown_output = briefing.get("markdown_output", "")

if markdown_output:
    display(Markdown(markdown_output))
else:
    # Fallback to render individual briefing fields
    md_blocks = [f"# 📋 Executive Briefing: {selected.get('title', 'Paper')}\n"]
    for key, val in briefing.items():
        if key != "markdown_output":
            title = key.replace("_", " ").title()
            md_blocks.append(f"### {title}\n{val}\n")
    display(Markdown("\n".join(md_blocks)))


---
## 7. Grounded QA Exploration & Anti-Hallucination Guardrails
The agent demonstrates:
1. **In-Scope Grounded Q&A**: Answers architectural and algorithmic questions with precise section citations (`[Section: ...]`).
2. **Anti-Hallucination Refusal**: When a question lies outside the paper's contents (e.g., Apple's stock price), the agent strictly refuses to hallucinate.
3. **Interactive Widget**: Ask any custom question about the paper in real time!


In [ ]:
# Helper function to submit QA questions to the LangGraph state machine
def query_agent(question: str) -> dict:
    qa_state = {
        "user_question": question,
        "session_id": SESSION_ID,
    }
    # Stream node execution
    for _ in graph.stream(qa_state, config=config, stream_mode="updates"):
        pass
    current_snap = graph.get_state(config)
    return current_snap.values.get("qa_answer", {})

print("=" * 75)
print("TEST 1: In-Scope Question (Core Architectural Innovation)")
print("=" * 75)
q1 = "What is the core architectural innovation of the Transformer?"
print(f"Q: {q1}\n")
ans1 = query_agent(q1)
display(Markdown(f"**Answer**:\n{ans1.get('answer', 'N/A')}"))
print(f"Citations: {ans1.get('citations', [])}\n")

print("=" * 75)
print("TEST 2: In-Scope Question (Attention Mechanisms)")
print("=" * 75)
q2 = "How does multi-head attention work and why is it beneficial?"
print(f"Q: {q2}\n")
ans2 = query_agent(q2)
display(Markdown(f"**Answer**:\n{ans2.get('answer', 'N/A')}"))
print(f"Citations: {ans2.get('citations', [])}\n")

print("=" * 75)
print("TEST 3: Out-of-Scope Question (Strict Anti-Hallucination Refusal)")
print("=" * 75)
q3 = "What was Apple's stock price on May 5, 2024?"
print(f"Q: {q3}\n")
ans3 = query_agent(q3)
display(Markdown(f"**Answer**:\n{ans3.get('answer', 'N/A')}"))
refusal_ok = "does not contain information" in ans3.get('answer', '').lower()
print(f"\nAnti-Hallucination Guardrail Triggered: {'✅ PASSED (Refused Ungrounded Query)' if refusal_ok else '❌ FAILED'}")


### 💡 Interactive Live QA Console
Type any question about the analyzed paper below and click **Ask Agent** or press **Enter**:


In [ ]:
txt_question = widgets.Text(
    value='',
    placeholder='e.g., What datasets were used for machine translation training?',
    description='Question:',
    layout=widgets.Layout(width='70%')
)

btn_ask = widgets.Button(
    description='Ask Agent',
    button_style='primary',
    icon='search'
)

qa_output = widgets.Output()

def on_submit_clicked(b):
    user_query = txt_question.value.strip()
    if not user_query:
        return
    with qa_output:
        qa_output.clear_output()
        print(f"🤔 Consulting paper digest for: '{user_query}'...")
        result = query_agent(user_query)
        qa_output.clear_output()
        display(Markdown(f"### Question\n> {user_query}\n"))
        display(Markdown(f"### Answer\n{result.get('answer', 'No response generated.')}\n"))
        citations = result.get('citations', [])
        if citations:
            display(Markdown(f"**Section Citations**: {", ".join(f'`{c}`' for c in citations)}"))

btn_ask.on_click(on_submit_clicked)
txt_question.on_submit(lambda _: on_submit_clicked(None))

display(widgets.VBox([
    widgets.HBox([txt_question, btn_ask]),
    qa_output
]))
